# Build static JSON

Reads `risk/mentions.csv` and `risk/ev_titles.csv` and writes three files under `output/`. The CSVs are not changed.

- `output/risks.json` — one object per risk id
- `output/publications.json` — one object per publication id
- `output/ev_ids.json` — one object per evidence id

Ids stay in first-seen order. `number_of_associated_records` counts mention rows for a risk. `number_of_associated_publications` counts distinct publication ids for a risk. `number_of_associated_ev_ids` counts distinct evidence ids for a risk or a publication. `number_of_associated_risks` counts distinct risk ids for a publication or an evidence id. Run the cells from the top.

In [2]:
import json
from pathlib import Path

import polars as pl

MENTIONS = Path("risk", "mentions.csv")
EV_TITLES = Path("risk", "ev_titles.csv")
OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

mentions = pl.read_csv(
    MENTIONS,
    schema_overrides={
        "id": pl.Int64,
        "label": pl.String,
        "ev_id": pl.String,
        "publication_id": pl.Int64,
    },
)
ev_titles = pl.read_csv(
    EV_TITLES,
    schema_overrides={
        "ev_id": pl.String,
        "publication_id": pl.Int64,
        "title": pl.String,
    },
)

def write_json(frame: pl.DataFrame, path: Path) -> None:
    path.write_text(json.dumps(frame.to_dicts(), indent=2) + "\n")
    print(f"{frame.height} records -> {path}")

empty_ids = pl.lit([], dtype=pl.List(pl.Int64))

In [3]:
risks = (
    mentions.group_by("id", maintain_order=True)
    .agg(
        pl.col("label").first(),
        pl.len().alias("number_of_associated_records"),
        pl.col("publication_id").unique(maintain_order=True),
        pl.col("ev_id").unique(maintain_order=True),
    )
    .rename({"id": "risk_id", "publication_id": "publication_ids", "ev_id": "ev_ids"})
    .with_columns(
        pl.col("publication_ids").list.len().alias("number_of_associated_publications"),
        pl.col("ev_ids").list.len().alias("number_of_associated_ev_ids"),
    )
    .select(
        "risk_id",
        "label",
        "number_of_associated_records",
        "number_of_associated_publications",
        "publication_ids",
        "number_of_associated_ev_ids",
        "ev_ids",
    )
)

write_json(risks, OUTPUT / "risks.json")

202 records -> output/risks.json


In [4]:
publication_risks = (
    mentions.group_by("publication_id")
    .agg(pl.col("id").unique(maintain_order=True).alias("risks"))
)
publications = (
    ev_titles.group_by("publication_id", maintain_order=True)
    .agg(
        pl.col("title").first(),
        pl.col("ev_id").alias("ev_ids"),
    )
    .join(publication_risks, on="publication_id", how="left")
    .with_columns(pl.col("risks").fill_null(empty_ids))
    .with_columns(
        pl.col("ev_ids").list.len().alias("number_of_associated_ev_ids"),
        pl.col("risks").list.len().alias("number_of_associated_risks"),
    )
    .select(
        "publication_id",
        "title",
        "number_of_associated_ev_ids",
        "ev_ids",
        "number_of_associated_risks",
        "risks",
    )
)

write_json(publications, OUTPUT / "publications.json")

30 records -> output/publications.json


In [8]:
evidence_risks = (
    mentions.group_by("ev_id")
    .agg(pl.col("id").unique(maintain_order=True).alias("risks"))
)
ev_ids = (
    ev_titles.select("ev_id", "publication_id")
    .join(evidence_risks, on="ev_id", how="left")
    .with_columns(pl.col("risks").fill_null(empty_ids))
    .with_columns(pl.col("risks").list.len().alias("number_of_associated_risks"))
    .select("ev_id", "publication_id", "number_of_associated_risks", "risks")
)

write_json(ev_ids, OUTPUT / "ev_ids.json")

146 records -> output/ev_ids.json
